# Dataset Summary QC

This notebook inventories raw TIFF files, joins them to the MF5v1 plate layout, checks observed-well channel/z completeness, and exports reusable CSV/JSON artifacts before downstream analysis begins.

## 1. Setup & Configuration

Change `data_root` for a new dataset. The defaults assume the notebook is run from the repository root or from the `notebooks/` directory.

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd

cwd = Path.cwd()
if (cwd / "src").exists():
    REPO_ROOT = cwd
elif (cwd.parent / "src").exists():
    REPO_ROOT = cwd.parent
else:
    raise RuntimeError("Could not locate repository root containing src/.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dataset_analysis import (
    DEFAULT_EXPECTED_Z_INDICES,
    DEFAULT_PROJECTION_Z_INDEX,
    build_completeness_table,
    build_dataset_inventory,
    build_dataset_summary,
    build_summary_table,
    find_dataset_issues,
    load_expected_channels,
    plot_channel_completeness,
    plot_control_distribution,
    plot_drug_distribution,
    plot_plate_coverage,
    plot_z_completeness,
    write_summary_json,
)

data_root = REPO_ROOT / "data" / "Plate 2426" / "BF Images"
layout_path = REPO_ROOT / "docs" / "layout" / "MF5v1_plate_layout.json"
wavelength_config_path = REPO_ROOT / "config" / "wavelength_config.yaml"
output_dir = REPO_ROOT / "notebooks" / "output" / "dataset_summary"

expected_z_indices = list(DEFAULT_EXPECTED_Z_INDICES)  # z1-z20
projection_z_index = DEFAULT_PROJECTION_Z_INDEX       # z0 projection

output_dir.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

print(f"Repository root: {REPO_ROOT}")
print(f"Data root: {data_root}")
print(f"Output directory: {output_dir}")

## 2. Dataset Discovery

Build a per-image inventory by parsing raw filenames with `ConfigurableFileHandler`, then join each observed well to the MF5v1 layout annotation.

In [ ]:
expected_channels = load_expected_channels(wavelength_config_path)
inventory = build_dataset_inventory(
    data_root=data_root,
    layout_path=layout_path,
    wavelength_config_path=wavelength_config_path,
)

inventory_path = output_dir / "dataset_inventory.csv"
inventory.to_csv(inventory_path, index=False)

print(f"Found {len(inventory):,} image files")
print(f"Expected channels: {expected_channels}")
print(f"Inventory written to: {inventory_path}")
display(inventory.head(20))

## 3. Summary & Observed-Subset QC

Completeness is checked per observed `(plate_id, time_point, well_id)`. Missing full-plate wells are not treated as errors for subset datasets.

In [ ]:
completeness = build_completeness_table(
    inventory,
    expected_channels=expected_channels,
    expected_z_indices=expected_z_indices,
    projection_z_index=projection_z_index,
)
issues = find_dataset_issues(
    inventory,
    expected_channels=expected_channels,
    expected_z_indices=expected_z_indices,
    projection_z_index=projection_z_index,
)
summary = build_dataset_summary(
    inventory,
    issues,
    expected_channels=expected_channels,
    expected_z_indices=expected_z_indices,
    projection_z_index=projection_z_index,
)

issues_path = output_dir / "dataset_issues.csv"
summary_path = output_dir / "dataset_summary.json"
issues.to_csv(issues_path, index=False)
write_summary_json(summary, summary_path)

display(build_summary_table(
    inventory,
    issues,
    expected_channels=expected_channels,
    expected_z_indices=expected_z_indices,
    projection_z_index=projection_z_index,
))

print(f"Issues written to: {issues_path}")
print(f"Summary written to: {summary_path}")

if not issues.empty and (issues["severity"] == "error").any():
    warnings.warn("Error-severity dataset issues were found. Review dataset_issues.csv before downstream analysis.")

## 4. Plate Coverage & Completeness Heatmaps

In [ ]:
if inventory.empty:
    print("No inventory rows available for heatmaps.")
else:
    contexts = inventory[["plate_id", "time_point"]].drop_duplicates().sort_values(["plate_id", "time_point"])
    for context in contexts.itertuples(index=False):
        plate_id = context.plate_id
        time_point = int(context.time_point)
        for figure in [
            plot_plate_coverage(inventory, plate_id=plate_id, time_point=time_point),
            plot_channel_completeness(completeness, expected_channels, plate_id=plate_id, time_point=time_point),
            plot_z_completeness(completeness, plate_id=plate_id, time_point=time_point),
        ]:
            plt.show()

## 5. Observed Control & Drug Distributions

In [ ]:
for figure in [plot_control_distribution(inventory), plot_drug_distribution(inventory)]:
    plt.show()

## 6. Missing Data & Anomaly Report

In [ ]:
if issues.empty:
    print("No dataset issues found for observed wells.")
else:
    issue_counts = issues.groupby(["severity", "issue_type"]).size().rename("count").reset_index()
    display(issue_counts)
    display(issues.sort_values(["severity", "plate_id", "time_point", "well_id", "issue_type", "z_index"]).head(200))

## Deferred Sections

Blur-score prescreening and cross-plate consistency checks are intentionally deferred until blur-cache and multi-plate folder policies are settled.